In [ ]:
import mne
import mne_bids
from mne_bids import BIDSPath
import os, os.path as op
import nilearn
import matplotlib.pyplot as plt
import numpy as np
import glob
import nih2mne

from nilearn import *

from nih2mne.utilities.bids_helpers import get_mri_dict
n_jobs = 10 #Number of parrallel operations

In [ ]:
os.environ["QT_API"] = "pyside6"
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
os.environ["PYVISTA_TRAME_JUPYTER_MODE"] = "server"

## Set some generic BIDS information

In [ ]:
#Raw data
bids_root = op.join('/data/', os.environ['USER'], 'meg_workshop_data')  
#Processed Data Folder - derivaties
deriv_root = op.join(bids_root, 'derivatives')  
project_root = op.join(deriv_root, 'Workshop')  
#Freesurfer brain surface reconstruction
fs_subjects_dir = subjects_dir = op.join(deriv_root, 'freesurfer','subjects')

### Find data for a subject

In [ ]:
# Let's loop over all the subjects and create a list of freesurfer subjects
subjects=['ON02811','ON03748','ON22671','ON42107','ON52662','ON61373','ON62003','ON70467','ON72082',
         'ON84896','ON85305','ON89474','ON89475']
fs_subjects=[]
for subject in subjects:
    fs_subjects.append('sub-' + subject)

#### Now, we are going to do exactly what we did in the MRI processing labs and the Source localization labs, but for every subject

In [ ]:
# declare a few empty lists to hold the source estimates

stcs_stim=[]
stcs_missing_stim=[]
stcs_contrast=[]

for subject in subjects:
    # filenames and load MRI files
    fs_subject = 'sub-' + subject
    bids_path = mne_bids.BIDSPath(root=bids_root, subject = subject, session='01', task='airpuff', run='01', 
                        datatype='meg')
    project_path = bids_path.copy().update(root=project_root, check=False)
    
    data_dict = nih2mne.utilities.bids_helpers.get_mri_dict(subject,bids_root, task='airpuff', project='Workshop')
    bem = data_dict['bem'].load()
    fwd = data_dict['fwd'].load()
    src = fwd['src']
    trans = data_dict['trans'].load()
    #Load the Raw data
    raw = mne.io.read_raw_ctf(bids_path.fpath, preload=True, verbose=False)
    noise_path = mne_bids.BIDSPath(root=bids_root, subject = subject, session='01', task='noise', run='01', 
                        datatype='meg')
    try:
        noise_fpath = noise_path.fpath
        if noise_fpath.exists():
            noise = mne.io.read_raw_ctf(noise_fpath, preload=True, verbose=False)
    except (FileNotFoundError, OSError, ValueError, RuntimeError) as e:
        print(f'sub-{subject}: could not load noise recording ({e})')
    
    # filter for beta        
    beta=raw.copy().filter(13,30)
    
    evts, evtsid = mne.events_from_annotations(raw)
    epochs_beta = mne.Epochs(beta, evts, evtsid, tmin=-0.1, tmax=0.3, preload=True)
    cov_all_beta = mne.compute_covariance(epochs_beta, tmin=0.15, tmax = 0.25, n_jobs=n_jobs)
    cov_stim_beta = mne.compute_covariance(epochs_beta['stim'], tmin=.15, tmax = 0.25, n_jobs=n_jobs)
    cov_missingstim_beta = mne.compute_covariance(epochs_beta['missingstim'], tmin=0.15, tmax = 0.25, n_jobs=n_jobs)
    if noise is not None:
        noise_beta=noise.copy().filter(13,30)
        noise_cov_beta = mne.compute_raw_covariance(noise_beta, tmin=0, tmax=None, method="empirical", rank=None)
    else:   
        noise_cov_beta = mne.make_ad_hoc_cov(epochs_beta.info,std=5.0e-13)        

    filters_beta = mne.beamformer.make_lcmv(epochs_beta.info, fwd, cov_all_beta, reg=0.05, pick_ori='max-power',
                                   noise_cov = noise_cov_beta, weight_norm='unit-noise-gain')
    stc_stim_beta = mne.beamformer.apply_lcmv_cov(cov_stim_beta, filters_beta)
    stc_stim_beta._data = stc_stim_beta._data.real
    stc_missing_beta = mne.beamformer.apply_lcmv_cov(cov_missingstim_beta, filters_beta)
    stc_missing_beta._data = stc_missing_beta._data.real
    stc_contrast=stc_stim_beta.copy()
    stc_contrast.data=np.log10(stc_stim_beta.data/stc_missing_beta.data)
    stcs_stim.append(stc_stim_beta)
    stcs_missing_stim.append(stc_missing_beta)
    stcs_contrast.append(stc_contrast)

In [ ]:
# you can look at any one of these - change the number and compare
# Note that I have the hemisphere set to 'lh' because that's where we saw the response; you can 
# set this to 'rh' or 'both' if you want to see the activity in both hemispheres. 

subjectsnum=6
brain=stcs_contrast[subjectsnum].plot(hemi='lh', subjects_dir=fs_subjects_dir, 
                                      subject=fs_subjects[subjectsnum])

In [ ]:
## In order to do any comparing between subjects, we have to put them in the same space
## FreeSurfer has the fsaverage space for this. 

fname_fsaverage_src = op.join(fs_subjects_dir,'fsaverage','bem','fsaverage-ico-5-src.fif')
src_to = mne.read_source_spaces(fname_fsaverage_src)

In [ ]:
# We'll make a new list for our morphed contrast images and populate it

stcs_contrast_morphed = []
for i in range(len(subjects)):
    # compute the morph for a given subject
    morph = mne.compute_source_morph(stcs_contrast[i], subject_from=fs_subjects[i], subject_to='fsaverage',
                                src_to=src_to, subjects_dir=fs_subjects_dir)
    # apply the morph
    stcs_contrast_morphed.append(morph.apply(stcs_contrast[i]))

In [ ]:
# just for visualization, lets make an average source estimate
src_avg_data = np.zeros(np.shape(stcs_contrast_morphed[0].data))
for i in range(len(subjects)):
    src_avg_data += stcs_contrast_morphed[i].data
src_avg=stcs_contrast_morphed[0].copy()
src_avg.data=(src_avg_data/len(subjects))

In [ ]:
# look at that estimate
brain = src_avg.plot(hemi='lh', subjects_dir=fs_subjects_dir, subject='fsaverage')

In [ ]:
## Wow! That looks pretty nice in the subject average. 

In [ ]:
## MNE python really only has pretty simple statistics for comparing groups. Exporting your surfaces
## and then working with afni is often the better choice, particularly when you have complicated designs
## where you might be comparing groups, conditions, using covariates, etc. But you can run simple stats, 
## like a one sample t-test. First let's set up the design matrix X:

# (n_subjects, n_vertices, n_times)
X = np.array([stc.data for stc in stcs_contrast_morphed])

In [ ]:
# MNE mainly uses permutation tests. For a one-sample t-test, for each permutation, mne randomly flips the 
# sign of each subject's data. It then re-computes the t-test statistic for the flipped dataset. It repeats
# this over and over, and builds up a null distribution for the t-statistic. It can then derive a p-value 
# reflecting how likely the actual t-stat at each vertice was to be as extreme by chance alone.

from mne.stats import permutation_t_test

t_obs, p_corr, H0 = permutation_t_test(X[:, :, 0], n_permutations=10000, tail=0)
# p_corr is already corrected across vertices

In [ ]:
## Note that we asked for 10,000 permutations, but it only did 4096 - this is because with 13 
## subejcts there are only so many permutations possible. 

In [ ]:
# That permutation test corrected for multiple comparisions. 
# We'll set our significace threshold at p < 0.05

sig = p_corr < 0.05
print(f'{sig.sum()} / {sig.size} vertices significant (corrected p < 0.05)')

# Now we'll mask our t-map to the significant vertices
stc_t = stcs_contrast_morphed[0].copy()
stc_t.data = np.where(sig, t_obs, 0)[:, np.newaxis]

In [ ]:
stc_t.plot(subject='fsaverage', subjects_dir=subjects_dir,
           hemi='split', views=['lateral', 'medial'],
           clim=dict(kind='value', pos_lims=[2, 3, 5]),
           transparent=True)

In [ ]:
## Well that's a lot fewer vertices in that left cluster in somatosensory cortex than we expected, isn't it. 
## Permutation tests do tend to be fairly conservative. In addition, we only had 13 subjects, which really 
## isn't that many. We would always recommend more than that. Finally, there's one last issue. That permutation
## test takes each vertex and treats it individually. 

In [ ]:
# MNE has another kind of permutation test - one based on something called threshold-free-cluster-enhancement, 
# or TFCE. In real brain data, each vertex isn't independent, there is substantial spatial correlation because
# of the spatial resolution of MEG being a little bit blurry. TFCE takes into accunt both the strengh of 
# effects, as well as how many neighbors show the same effect. So it can be more sensitive. 

from mne.stats import permutation_cluster_1samp_test

fs_src = mne.read_source_spaces(
    op.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-5-src.fif'))
adjacency = mne.spatial_src_adjacency(fs_src)

X2 = X[:, :, 0]    # (n_subjects, n_vertices)

# TFCE: no fixed cluster-forming threshold
t_tfce, clusters, p_tfce, H0 = permutation_cluster_1samp_test(
    X2, adjacency=adjacency, threshold=dict(start=0, step=0.2),
    n_permutations=10000, tail=0, n_jobs=n_jobs)

In [ ]:
sig_tfce = p_tfce < 0.05
print(sig_tfce.sum(), 'vertices significant with TFCE')

In [ ]:
## Wow, that's a lot more significant vertices

In [ ]:
stc_tfce = stcs_contrast_morphed[0].copy()
stc_tfce.data = np.where(sig_tfce, t_tfce, 0)[:, np.newaxis]

lim = np.abs(t_tfce[sig_tfce]).max()
stc_tfce.plot(
    subject='fsaverage', subjects_dir=subjects_dir,
    hemi='split', views=['lateral', 'medial'],
    clim=dict(kind='value', pos_lims=[0.1 * lim, 0.4 * lim, lim]),
    transparent=True)

In [ ]:
## As you can see, threshold free clustering is powerful. But again, the resources in 
## MNE are still limited for complex experimental designs. Note that we aren't covering writing these
## surface maps out for export to afni, and part of that is because we've used a pretty course surface
## mesh. We can show you how to do this, but we aren't going to cover it in the course. 

## Parcellation

In [ ]:
### Another thing we can do is use cortical parcellations to look at group data. This is often useful
### because it reduces the number of statistical tests from all the vertices to a limited number of 
### vertices.

In [ ]:
# I'm goign to use a pretty fine parcellation called aparc_sub - it has 448 total parcels, approx 224 in 
# each hemisphere. I'll extract the labels from the fsaverage template brain surface. 

labels = mne.read_labels_from_annot('fsaverage', parc='aparc_sub', subjects_dir=subjects_dir)
labels = [l for l in labels if not l.name.startswith('unknown')]   # drop non-cortex label

# Read in the freesurfer fsaverage source space
fs_src = mne.read_source_spaces(
    op.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-5-src.fif'))
# Remember that stcs_contrast_morphed is our list of all the contrast stc's for all the subjects morphed
# to common space. We are going to extract the values for each label for each subject
tcs = mne.extract_label_time_course(
    stcs_contrast_morphed, labels, fs_src, mode='mean')    # list of (n_labels, n_times)

# This just sets up an array that is n_subjects x n_labels
X_lab = np.array([tc[:, 0] for tc in tcs])    # (n_subjects, n_labels), single time point
# Get the label names
label_names = [l.name for l in labels]
# Now get a group mean within the labels
mean_lab = X_lab.mean(axis=0)                 # group mean per parcel

# That mean_lab variable is still just a numpy array - for mne  to want to plot it we have to put it into
# the stc structure
stc_mean = mne.labels_to_stc(labels, mean_lab, subject='fsaverage')

In [ ]:
# Now plot the mean across subjects, but this time using the parcel means

lim = np.abs(mean_lab).max()
stc_mean.plot(
    subject='fsaverage', subjects_dir=subjects_dir,
    hemi='split', views=['lateral', 'medial'],
    clim=dict(kind='value', pos_lims=[0.1 * lim, 0.5 * lim, lim]),
    transparent=False)    

In [ ]:
# You can see there are multiple parcels with large means.
# Let's plot the values in those parcels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let's take the top 5 parcels, and we'll rank by absolute value - so we allow positive and negative means
n_top = 5   
by_abs = True

rank_vals = np.abs(mean_lab) if by_abs else mean_lab
top = np.argsort(rank_vals)[::-1][:n_top]          # extract the indices of top parcels

sem = X_lab.std(axis=0, ddof=1) / np.sqrt(X_lab.shape[0])  # get the standard error of the mean

fig, ax = plt.subplots(figsize=(7, 4.5))
rng = np.random.default_rng(0)  ## this is just a random number generator to shift all the points a little bit left and right
                                ## to make them easier to see. Seaborn has nicer graphs that do this automatically. 

ax.axhline(0, color='k', lw=0.8, ls='--')   # This just plots a line at zero, since this is a one sample t-test we are testing
                                            # Whether our data is different from zero
for x, i in enumerate(top):  # Loop over parcels and add them to the graph
    # one dot per subject, with a little horizontal jitter so they don't overlap
    jitter = rng.uniform(-0.15, 0.15, size=X_lab.shape[0])
    ax.scatter(np.full(X_lab.shape[0], x) + jitter, X_lab[:, i],       # First do the scatter plot
               s=25, color='gray', alpha=0.6, zorder=2)
    # group mean with SEM error bar
    ax.errorbar(x, mean_lab[i], yerr=sem[i], fmt='o', color='crimson', # then add the means and standard errors
                markersize=9, capsize=6, linewidth=2, zorder=3)

In [ ]:
# You can see that there is a lot of scatter across subjects! Some subjects show a relatively large effect, 
# others not so much. If you are interested, we can add labels by subject

fig, ax = plt.subplots(figsize=(8, 5))
for x, i in enumerate(top):
    xs = x + rng.uniform(-0.2, 0.2, size=X_lab.shape[0])
    ax.scatter(xs, X_lab[:, i], s=25, color='gray', alpha=0.7, zorder=2)

    # label each dot with its subject ID
    for xj, yj, subj in zip(xs, X_lab[:, i], subjects):
        ax.annotate(subj[-5:], (xj, yj), fontsize=6,       # last 3 characters of the ID
                    xytext=(3, 2), textcoords='offset points')

    ax.errorbar(x, mean_lab[i], yerr=sem[i], fmt='D', color='crimson',
                markersize=8, capsize=6, linewidth=2, zorder=3)

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(range(len(top)))
ax.set_xticklabels([label_names[i] for i in top], rotation=30, ha='right')
ax.set_ylabel('log10(stim / missingstim) beta power')
plt.tight_layout()
plt.show()

In [ ]:
# Funny, turns out the first subject in the dataset is the one that has the largest effect! 

#### Exercise 1

1. Not sure if you remember at this point, but when we did the time frequency plot there was also an effect in theta around .045s, and an effect in alpha around 0.1 s. You could copy cell #13 from above and run for alpha or theta, changing the time periods. The morphs to fsaverage are already calculated so you don't need to re-calculated, you can just apply the morphs to your new stc contrasts lists. Then you could calculated the means, or even do permutation tests and see what you see! 